In [1]:
import queue
import sounddevice as sd
import json
from vosk import Model, KaldiRecognizer
import ipywidgets as widgets
from IPython.display import display
from threading import Thread

# Configuration
SAMPLE_RATE = 44100   # recommended sample rate for Vosk models :contentReference[oaicite:2]{index=2}
CHANNELS = 1          # mono audio :contentReference[oaicite:3]{index=3}

# Audio queue: streaming callback will push raw bytes here
audio_q = queue.Queue()
model_path = "vosk-model-small-en-us-0.15/vosk-model-small-en-us-0.15"

# Load model + recognizer
model = Model(model_path)  # change to your actual model path
rec = KaldiRecognizer(model, SAMPLE_RATE)
rec.SetWords(True)

# Flag to control streaming
listening = {"active": False}

def speach_converter(text):
    pass

def audio_callback(indata, frames, time, status):
    """sounddevice callback — push audio data to queue."""
    if status:
        print("Audio status:", status)
    audio_q.put(bytes(indata))

def recognize_loop(output_widget):
    """Continuously read from queue and feed to Vosk recognizer."""
    while listening["active"]:
        try:
            data = audio_q.get(timeout=0.1)
        except queue.Empty:
            continue
        if rec.AcceptWaveform(data):
            result_json = json.loads(rec.FinalResult())
            text = result_json.get("text", "")
            if text:
                with output_widget:
                    print(text)

def start_stream(_):
    if listening["active"]:
        return  # already running
    listening["active"] = True
    # Start sounddevice input stream
    stream = sd.RawInputStream(samplerate=SAMPLE_RATE,
                               blocksize = 8000,
                               dtype='int16',
                               channels=CHANNELS,
                               callback=audio_callback)
    stream.start()
    # Start recognizer thread
    t = Thread(target=recognize_loop, args=(output,), daemon=True)
    t.start()
    # Store stream so we can stop it later
    listening["stream"] = stream
    with output:
        print("[Listening started …]")

def stop_stream(_):
    if not listening["active"]:
        return
    listening["active"] = False
    # Stop and close the sound stream
    stream = listening.get("stream")
    if stream:
        stream.stop()
        stream.close()
    with output:
        print("[Listening stopped]")

# Setup widgets
record_button = widgets.Button(description="Start Recording", button_style="success", icon="microphone")
stop_button   = widgets.Button(description="Stop", button_style="danger", icon="stop")
output = widgets.Output(layout={'border': '1px solid black'})

record_button.on_click(start_stream)
stop_button.on_click(stop_stream)

display(record_button, stop_button, output)


Button(button_style='success', description='Start Recording', icon='microphone', style=ButtonStyle())

Button(button_style='danger', description='Stop', icon='stop', style=ButtonStyle())

Output(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='1px solid b…

playing the game
you my name is john
hello i'm happy need
i'm happy to meet you
hello hashem
this is a message for you
try to
record the voice
using your microphone
